<a href="https://colab.research.google.com/github/anshupandey/EY_GenAI_Architect/blob/main/Task_market_profiling_agent_langgraph_azure_openai_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Market Profiling Agent using LangGraph, LangChain, and Azure OpenAI

## Background

In financial decision-making, investors often rely on **recent news sentiment and market context** before deciding whether to invest in a company. Manually gathering this information can be time‑consuming.

In this task, we design a **workflow-based AI agent** that automates this process.

The agent receives a **company name** as input and performs the following steps:

1. Gather recent news about the company from **Yahoo Finance**.
2. Perform **sentiment analysis** on the gathered news.
3. If sentiment is **positive**, the agent generates investment suggestions.
4. If sentiment is **negative**, the agent gathers additional context using **Tavily Search** to understand the reasons behind the negative outlook.
5. Based on the additional research, the agent generates **recommendations and feedback**.

This design demonstrates how **LangGraph** can orchestrate multi-step reasoning workflows with tools.

---

## Workflow Overview

Start → Input Company Name → Fetch Yahoo Finance News → Sentiment Check

If Positive:
→ Generate Suggestions → END

If Negative:
→ Tavily Search (market research)
→ Generate Recommendations and Feedback → END

---

## Technologies Used

- **LangChain** – Tool integration and LLM abstraction
- **LangGraph** – Workflow orchestration for agents
- **Azure OpenAI** – LLM for reasoning and generation
- **Yahoo Finance / yfinance** – Financial news source
- **Tavily Search** – Internet search for additional research

---


## Install Dependencies

In [1]:

# Uncomment if running locally
!pip install langchain langgraph langchain-openai yfinance tavily-python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 12.6 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.17
    Uninstalling langchain-core-1.2.17:
      Successfully uninstalled langchain-core-1.2.17


## Import Libraries

In [3]:

import os
import yfinance as yf

from langchain.chat_models import init_chat_model
from tavily import TavilyClient

from langgraph.graph import StateGraph, END
from typing import TypedDict


## Configure Azure OpenAI

In [ ]:

llm = init_chat_model(model = "azure_openai:gpt-4o-mini")


## Define State for the Workflow

In [4]:

class AgentState(TypedDict):
    company: str
    news: str
    sentiment: str
    research: str
    output: str


## Yahoo Finance News Tool

In [5]:

def get_yahoo_news(company):

    ticker = yf.Ticker(company)
    news_items = ticker.news

    texts = []
    for item in news_items[:5]:
        texts.append(item["title"])

    return " ".join(texts)


## Tavily Search Tool

In [ ]:

tavily = TavilyClient()

def tavily_search(company):

    result = tavily.search(query=f"{company} negative news investment risks", max_results=5)

    texts = []
    for r in result["results"]:
        texts.append(r["content"])

    return " ".join(texts)


## Node 1 – Gather Yahoo Finance News

In [ ]:

def gather_news(state):

    company = state["company"]
    news = get_yahoo_news(company)

    return {"news": news}


## Node 2 – Sentiment Analysis

In [ ]:

def analyze_sentiment(state):

    news = state["news"]

    prompt = f"""
    Determine whether the sentiment of the following news is Positive or Negative.

    News:
    {news}

    Return only one word: Positive or Negative
    """

    result = llm.invoke(prompt)

    return {"sentiment": result.content.strip()}


## Node 3 – Market Research (Negative Case)

In [ ]:

def market_research(state):

    company = state["company"]

    research = tavily_search(company)

    return {"research": research}


## Node 4 – Generate Suggestions (Positive Case)

In [ ]:

def generate_positive(state):

    news = state["news"]

    prompt = f"""
    Based on the following positive news, suggest an investment strategy.

    News:
    {news}
    """

    response = llm.invoke(prompt)

    return {"output": response.content}


## Node 5 – Generate Recommendations (Negative Case)

In [ ]:

def generate_negative(state):

    research = state["research"]

    prompt = f"""
    The company shows negative sentiment.

    Using the research below, explain the risks and provide investment feedback.

    Research:
    {research}
    """

    response = llm.invoke(prompt)

    return {"output": response.content}


## Build LangGraph Workflow

In [ ]:

workflow = StateGraph(AgentState)

workflow.add_node("gather_news", gather_news)
workflow.add_node("sentiment", analyze_sentiment)
workflow.add_node("research", market_research)
workflow.add_node("positive", generate_positive)
workflow.add_node("negative", generate_negative)

workflow.set_entry_point("gather_news")

workflow.add_edge("gather_news", "sentiment")

def route(state):
    if "positive" in state["sentiment"].lower():
        return "positive"
    return "research"

workflow.add_conditional_edges(
    "sentiment",
    route,
    {
        "positive": "positive",
        "research": "research"
    }
)

workflow.add_edge("research", "negative")
workflow.add_edge("positive", END)
workflow.add_edge("negative", END)

graph = workflow.compile()


## Run the Agent

In [ ]:

result = graph.invoke({
    "company": "Tesla"
})

print(result["output"])
